# Legal Document Summarizer - Upload & OCR Version
## Support for PDF, DOCX, Images with OCR for Scanned Documents

**Features:**
- Upload PDFs, DOCX, TXT, and image files (PNG, JPG)
- Automatic OCR for scanned/image documents
- Multi-agent legal analysis
- Quality evaluation
- Export results

In [1]:
# Installation - Run once
!pip install -q langchain langchain-core langgraph pydantic langchain-groq
!pip install -q pypdf python-docx Pillow
!pip install -q pytesseract pdf2image  # OCR support
!pip install -q gradio  # For file upload interface (optional)
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr

# Note: On Linux, you may need to install system dependencies:
# !apt-get install -y tesseract-ocr poppler-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 9.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.8 MB/s eta 0:00:00a 0:00:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 124463 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
# Imports
import os
import io
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime
from typing import TypedDict, Dict, List, Optional

from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# Document processing
from pypdf import PdfReader
from docx import Document
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
# Configuration
GROQ_API_KEY = "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"  # Replace with your key
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Model configuration
PRIMARY_MODEL = "llama-3.3-70b-versatile"  # Best quality
FALLBACK_MODEL = "llama-3.1-8b-instant"     # Fast, for rate limits

# Processing settings
MAX_RETRIES = 3
RETRY_DELAY = 2
OCR_ENABLED = True  # Enable/disable OCR
OCR_LANGUAGE = "eng"  # Tesseract language (eng, ara, urd, etc.)

# File upload settings
BASE_DIR = "/kaggle/working"

UPLOAD_FOLDER = os.path.join(BASE_DIR, "up")
OUTPUT_FOLDER = os.path.join(BASE_DIR, "outputs")

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"✅ Configuration complete")
print(f"📁 Upload folder: {UPLOAD_FOLDER}")
print(f"📁 Output folder: {OUTPUT_FOLDER}")
print(f"🔍 OCR enabled: {OCR_ENABLED}")

✅ Configuration complete
📁 Upload folder: /kaggle/working/up
📁 Output folder: /kaggle/working/outputs
🔍 OCR enabled: True


## Enhanced Document Extraction with OCR

In [4]:
class DocumentExtractionResult(BaseModel):
    """Result of document text extraction"""
    text: str
    extraction_method: str  # 'text', 'ocr', 'hybrid'
    file_type: str
    page_count: int
    confidence: float  # OCR confidence if applicable
    warnings: List[str] = []

def classify_document(state):
    """FIXED: Classification with safe dictionary access"""
    import time
    start_time = time.time()
    print("📋 [Classifier] Analyzing document type...")
    
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    import json
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal document classification expert. Analyze this document and classify it.
        
        Return ONLY a JSON object (no markdown, no backticks) with these exact fields:
        {{
            "document_type": "court_opinion",
            "confidence": 0.85,
            "jurisdiction": "Pakistan Supreme Court",
            "key_indicators": ["Civil Appeal", "Supreme Court", "judgment"],
            "suggested_focus": "Analyze the court's reasoning and legal principles"
        }}
        
        Document types: court_opinion, contract, statute, brief, regulation, other
        
        DOCUMENT TEXT (first 3000 chars):
        {document_text}
        """
    )
    
    # Assuming llm is already defined
    chain = prompt | llm | StrOutputParser()
    
    try:
        response = invoke_with_retry(chain, {
            "document_text": state["document_text"][:3000]
        })
        
        # Clean and parse JSON
        clean_response = response.strip()
        if clean_response.startswith("```json"):
            clean_response = clean_response[7:]
        if clean_response.startswith("```"):
            clean_response = clean_response[3:]
        if clean_response.endswith("```"):
            clean_response = clean_response[:-3]
        clean_response = clean_response.strip()
        
        try:
            result = json.loads(clean_response)
        except json.JSONDecodeError:
            print(f"⚠️  JSON parse failed, using defaults")
            result = {}
        
        # Safe access with defaults
        state["classification"] = result.get("document_type", "unknown")
        state["classification_confidence"] = float(result.get("confidence", 0.0))
        state["document_metadata"] = {
            "jurisdiction": result.get("jurisdiction", "Unknown"),
            "key_indicators": ", ".join(result.get("key_indicators", [])),
            "suggested_focus": result.get("suggested_focus", "General legal analysis")
        }
        
        print(f"✅ Classification: {state['classification']} (confidence: {state['classification_confidence']:.2f})")
        
    except Exception as e:
        print(f"❌ Classification error: {e}")
        state["classification"] = "unknown"
        state["classification_confidence"] = 0.0
        state["document_metadata"] = {
            "jurisdiction": "Unknown",
            "key_indicators": "Classification error",
            "suggested_focus": "General legal analysis"
        }
        state["errors"].append(f"Classification failed: {str(e)}")
    
    state["processing_time"]["classification"] = time.time() - start_time
    return state


def is_scanned_pdf(pdf_path: str, sample_pages: int = 3) -> bool:
    """Detect if PDF is scanned (image-based) or has extractable text"""
    try:
        reader = PdfReader(pdf_path)
        pages_to_check = min(sample_pages, len(reader.pages))
        
        total_text_length = 0
        for i in range(pages_to_check):
            text = reader.pages[i].extract_text()
            total_text_length += len(text.strip())
        
        # If average text per page is very low, it's likely scanned
        avg_text_per_page = total_text_length / pages_to_check
        return avg_text_per_page < 100  # Threshold for scanned detection
        
    except Exception as e:
        print(f"⚠️  Error checking if PDF is scanned: {e}")
        return True  # Assume scanned if error

def extract_text_from_image(image_path: str, language: str = OCR_LANGUAGE) -> tuple[str, float]:
    """Extract text from image using OCR"""
    try:
        image = Image.open(image_path)
        
        # Get detailed OCR data for confidence score
        ocr_data = pytesseract.image_to_data(image, lang=language, output_type=pytesseract.Output.DICT)
        
        # Extract text
        text = pytesseract.image_to_string(image, lang=language)
        
        # Calculate average confidence
        confidences = [float(conf) for conf in ocr_data['conf'] if conf != '-1']
        avg_confidence = sum(confidences) / len(confidences) if confidences else 0.0
        
        return text, avg_confidence / 100  # Normalize to 0-1
        
    except Exception as e:
        raise Exception(f"OCR failed on image: {e}")

def extract_text_from_scanned_pdf(pdf_path: str, language: str = OCR_LANGUAGE) -> tuple[str, float]:
    """Extract text from scanned PDF using OCR"""
    try:
        print("🔍 Performing OCR on scanned PDF (this may take a while)...")
        
        # Convert PDF pages to images
        images = convert_from_path(pdf_path, dpi=300)
        
        all_text = []
        all_confidences = []
        
        for i, image in enumerate(images):
            print(f"   Processing page {i+1}/{len(images)}...")
            
            # Perform OCR on each page
            ocr_data = pytesseract.image_to_data(image, lang=language, output_type=pytesseract.Output.DICT)
            page_text = pytesseract.image_to_string(image, lang=language)
            
            all_text.append(page_text)
            
            # Calculate page confidence
            confidences = [float(conf) for conf in ocr_data['conf'] if conf != '-1']
            if confidences:
                all_confidences.append(sum(confidences) / len(confidences))
        
        combined_text = "\n\n".join(all_text)
        avg_confidence = sum(all_confidences) / len(all_confidences) if all_confidences else 0.0
        
        return combined_text, avg_confidence / 100
        
    except Exception as e:
        raise Exception(f"OCR failed on PDF: {e}")

def extract_text_from_file(file_path: str, enable_ocr: bool = OCR_ENABLED) -> DocumentExtractionResult:
    """Extract text from various file formats with OCR support"""
    
    file_path = Path(file_path)
    file_ext = file_path.suffix.lower()
    
    print(f"\n📄 Processing: {file_path.name}")
    print(f"   File type: {file_ext}")
    
    result = DocumentExtractionResult(
        text="",
        extraction_method="unknown",
        file_type=file_ext,
        page_count=0,
        confidence=1.0,
        warnings=[]
    )
    
    try:
        # Text files
        if file_ext == '.txt':
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                result.text = f.read()
            result.extraction_method = 'text'
            result.page_count = 1
            print("   ✅ Text extracted directly")
        
        # PDF files
        elif file_ext == '.pdf':
            # First, try to extract text directly
            reader = PdfReader(str(file_path))
            result.page_count = len(reader.pages)
            
            # Check if PDF is scanned
            is_scanned = is_scanned_pdf(str(file_path))
            
            if is_scanned and enable_ocr:
                print("   📸 Detected scanned PDF, using OCR...")
                result.text, result.confidence = extract_text_from_scanned_pdf(str(file_path))
                result.extraction_method = 'ocr'
                
                if result.confidence < 0.7:
                    result.warnings.append(f"Low OCR confidence: {result.confidence:.1%}")
            
            else:
                # Extract text directly
                text_parts = []
                for page in reader.pages:
                    text_parts.append(page.extract_text())
                result.text = "\n\n".join(text_parts)
                result.extraction_method = 'text'
                
                # If very little text was extracted but OCR is disabled
                if len(result.text.strip()) < 500 and not enable_ocr:
                    result.warnings.append("Low text content detected. Consider enabling OCR.")
            
            print(f"   ✅ Extracted {len(result.text)} characters from {result.page_count} pages")
        
        # Word documents
        elif file_ext in ['.doc', '.docx']:
            doc = Document(str(file_path))
            result.text = "\n".join([para.text for para in doc.paragraphs])
            result.extraction_method = 'text'
            result.page_count = len(doc.sections)
            print("   ✅ Text extracted from Word document")
        
        # Image files
        elif file_ext in ['.png', '.jpg', '.jpeg', '.tiff', '.bmp']:
            if not enable_ocr:
                raise Exception("OCR is disabled. Cannot extract text from images.")
            
            print("   📸 Performing OCR on image...")
            result.text, result.confidence = extract_text_from_image(str(file_path))
            result.extraction_method = 'ocr'
            result.page_count = 1
            
            if result.confidence < 0.7:
                result.warnings.append(f"Low OCR confidence: {result.confidence:.1%}")
            
            print(f"   ✅ OCR complete (confidence: {result.confidence:.1%})")
        
        else:
            raise ValueError(f"Unsupported file type: {file_ext}")
        
        # Validate extraction
        if len(result.text.strip()) < 50:
            result.warnings.append("Very little text extracted. File may be empty or corrupted.")
        
        # Display warnings
        for warning in result.warnings:
            print(f"   ⚠️  {warning}")
        
        return result
        
    except Exception as e:
        raise Exception(f"Failed to extract text from {file_path.name}: {str(e)}")

## Core LLM Functions (with Retry Logic)

In [5]:
def get_llm(use_fallback=False, temperature=0):
    """Get LLM with fallback support"""
    model = FALLBACK_MODEL if use_fallback else PRIMARY_MODEL
    return ChatGroq(
        model=model,
        temperature=temperature,
        max_retries=MAX_RETRIES,
        timeout=60
    )

def invoke_with_retry(chain, inputs, max_retries=MAX_RETRIES):
    """Invoke chain with automatic retry and fallback"""
    for attempt in range(max_retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            error_msg = str(e)
            
            if "rate_limit" in error_msg.lower() or "429" in error_msg:
                print(f"⚠️  Rate limit hit on attempt {attempt + 1}/{max_retries}")
                
                if attempt < max_retries - 1:
                    wait_time = RETRY_DELAY * (2 ** attempt)
                    print(f"   Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                    continue
                else:
                    raise Exception(f"Rate limit exceeded after {max_retries} attempts")
            else:
                raise e
    
    raise Exception("Max retries reached")

# Initialize primary LLM
llm = get_llm(temperature=0)
print("✅ LLM initialized")

✅ LLM initialized


## State Definition & Agent Functions
*(Copy from previous improved version)*

In [6]:
class LegalDocumentState(TypedDict):
    """State with document metadata"""
    document_text: str
    document_metadata: Dict[str, str]
    extraction_info: Dict[str, any]  # NEW: OCR confidence, method, etc.
    classification: str
    classification_confidence: float
    extracted_facts: str
    extracted_issues: str
    extracted_holding: str
    extracted_reasoning: str
    extracted_ratio: str
    final_memo: str
    evaluation_score: Dict
    errors: List[str]
    processing_time: Dict[str, float]

In [7]:
def extract_facts(state: LegalDocumentState) -> LegalDocumentState:
    """Extract key facts with improved legal context"""
    start_time = time.time()
    print("📝 [Facts Agent] Extracting key facts...")
    
    doc_type = state.get("classification", "unknown")
    
    # Customize prompt based on document type
    if doc_type == "court_opinion":
        focus = """Focus on:
        - Parties involved (plaintiff/defendant, appellant/appellee)
        - Procedural history (lower court decisions)
        - Relevant factual background (what happened)
        - Material facts that influenced the decision
        - Timeline of events if relevant"""
    elif doc_type == "contract":
        focus = """Focus on:
        - Parties to the agreement
        - Subject matter and consideration
        - Key terms and conditions
        - Performance obligations
        - Duration and termination provisions"""
    else:
        focus = "Focus on key factual elements relevant to the legal context."
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal research assistant extracting FACTS from a {doc_type}.
        
        {focus}
        
        Be concise but comprehensive. Use legal terminology appropriately.
        Organize chronologically or by importance.
        Cite specific details (dates, amounts, locations) when present.
        
        DOCUMENT:
        {document_text}
        
        EXTRACTED FACTS:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["extracted_facts"] = invoke_with_retry(chain, {
            "doc_type": doc_type,
            "focus": focus,
            "document_text": state["document_text"][:8000]
        })
        print("✅ Facts extracted successfully")
    except Exception as e:
        print(f"❌ Facts extraction error: {e}")
        state["extracted_facts"] = "Error extracting facts"
        state["errors"].append(f"Facts extraction failed: {str(e)}")
    
    state["processing_time"]["facts"] = time.time() - start_time
    return state

def extract_issues(state: LegalDocumentState) -> LegalDocumentState:
    """Extract legal issues with proper framing"""
    start_time = time.time()
    print("⚖️  [Issues Agent] Identifying legal issues...")
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal research assistant identifying LEGAL ISSUES.
        
        Document Type: {doc_type}
        
        A well-framed legal issue:
        1. States the specific legal question to be resolved
        2. References applicable law/doctrine when clear
        3. Is phrased as a question or "whether" statement
        4. Focuses on legal principles, not just facts
        
        Examples:
        - "Whether the plaintiff's sale deed dated 15.01.1978 establishes superior title over defendants' 1971 oral sale and mutation under the Specific Relief Act."
        - "Whether the 12-year delay bars the claim under the doctrine of laches."
        
        For court opinions: Identify the questions the court addressed.
        For contracts: Identify potential areas of dispute or ambiguity.
        For statutes: Identify the legal problems being addressed.
        
        FACTS:
        {facts}
        
        FULL DOCUMENT:
        {document_text}
        
        LEGAL ISSUES:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["extracted_issues"] = invoke_with_retry(chain, {
            "doc_type": state.get("classification", "unknown"),
            "facts": state.get("extracted_facts", ""),
            "document_text": state["document_text"][:8000]
        })
        print("✅ Issues identified successfully")
    except Exception as e:
        print(f"❌ Issues extraction error: {e}")
        state["extracted_issues"] = "Error extracting issues"
        state["errors"].append(f"Issues extraction failed: {str(e)}")
    
    state["processing_time"]["issues"] = time.time() - start_time
    return state

def extract_holding(state: LegalDocumentState) -> LegalDocumentState:
    """Extract holding/decision"""
    start_time = time.time()
    print("🔨 [Holding Agent] Extracting court's decision...")
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal research assistant extracting the HOLDING/DECISION.
        
        For court opinions:
        - State the court's ultimate decision (affirmed, reversed, remanded, etc.)
        - Include specific relief granted or denied
        - Note any dissenting/concurring opinions if present
        
        For contracts:
        - Summarize the agreement reached
        - Highlight key obligations and rights created
        
        For statutes:
        - Summarize what the law establishes or prohibits
        - Note effective dates and scope
        
        Be specific and precise. Use legal terminology correctly.
        
        LEGAL ISSUES:
        {issues}
        
        DOCUMENT:
        {document_text}
        
        HOLDING/DECISION:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["extracted_holding"] = invoke_with_retry(chain, {
            "issues": state.get("extracted_issues", ""),
            "document_text": state["document_text"][:8000]
        })
        print("✅ Holding extracted successfully")
    except Exception as e:
        print(f"❌ Holding extraction error: {e}")
        state["extracted_holding"] = "Error extracting holding"
        state["errors"].append(f"Holding extraction failed: {str(e)}")
    
    state["processing_time"]["holding"] = time.time() - start_time
    return state

def extract_reasoning(state: LegalDocumentState) -> LegalDocumentState:
    """Extract court's reasoning"""
    start_time = time.time()
    print("🧠 [Reasoning Agent] Analyzing legal reasoning...")
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal research assistant extracting REASONING/ANALYSIS.
        
        Explain HOW and WHY the decision was reached:
        - Legal principles applied
        - Statutes or precedents cited
        - Logical steps in the analysis
        - Factual findings that supported the conclusion
        - Policy considerations if discussed
        
        Focus on the LEGAL reasoning, not just restating facts.
        Cite specific cases or statutes mentioned.
        Highlight key distinctions or interpretations made.
        
        HOLDING:
        {holding}
        
        DOCUMENT:
        {document_text}
        
        COURT'S REASONING:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["extracted_reasoning"] = invoke_with_retry(chain, {
            "holding": state.get("extracted_holding", ""),
            "document_text": state["document_text"][:10000]
        })
        print("✅ Reasoning extracted successfully")
    except Exception as e:
        print(f"❌ Reasoning extraction error: {e}")
        state["extracted_reasoning"] = "Error extracting reasoning"
        state["errors"].append(f"Reasoning extraction failed: {str(e)}")
    
    state["processing_time"]["reasoning"] = time.time() - start_time
    return state

def extract_ratio(state: LegalDocumentState) -> LegalDocumentState:
    """Extract ratio decidendi (binding legal principle)"""
    start_time = time.time()
    print("📜 [Ratio Agent] Identifying binding legal principle...")
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a legal scholar extracting the RATIO DECIDENDI (binding rule of law).
        
        The ratio decidendi is:
        - The legal principle essential to the decision
        - The rule that future courts must follow (precedent)
        - Stated as a general principle, not tied to specific facts
        - Distinguishable from obiter dicta (non-binding remarks)
        
        Format as: "[General legal principle] when [relevant circumstances]."
        
        Example: "A plaintiff must prove their case through valid and reasonable evidence when claiming property rights, and pleadings alone cannot substitute for evidence."
        
        REASONING:
        {reasoning}
        
        HOLDING:
        {holding}
        
        RATIO DECIDENDI:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["extracted_ratio"] = invoke_with_retry(chain, {
            "reasoning": state.get("extracted_reasoning", ""),
            "holding": state.get("extracted_holding", "")
        })
        print("✅ Ratio decidendi identified successfully")
    except Exception as e:
        print(f"❌ Ratio extraction error: {e}")
        state["extracted_ratio"] = "Error extracting ratio"
        state["errors"].append(f"Ratio extraction failed: {str(e)}")
    
    state["processing_time"]["ratio"] = time.time() - start_time
    return state


In [8]:
def synthesize_memo(state: LegalDocumentState) -> LegalDocumentState:
    """Synthesize all extractions into structured legal memo"""
    start_time = time.time()
    print("📝 [Synthesizer] Creating final legal memo...")
    
    doc_type = state.get("classification", "unknown")
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a senior law clerk preparing a professional case brief for an attorney or law student.
        
        Document Type: {doc_type}
        Jurisdiction: {jurisdiction}
        
        Synthesize the following extracted information into a clear, structured legal memo.
        Use proper legal writing conventions:
        - Be precise and concise
        - Use proper citations if case names are mentioned
        - Organize logically with numbered sections
        - Use formal legal language
        - Highlight key terms
        
        REQUIRED STRUCTURE FOR COURT OPINIONS:
        1. FACTS - Concise summary of relevant facts
        2. PROCEDURAL HISTORY - How the case reached this court
        3. LEGAL ISSUE(S) - Question(s) before the court
        4. HOLDING - Court's decision
        5. REASONING - Court's analysis and logic
        6. RATIO DECIDENDI - Binding legal principle
        7. SIGNIFICANCE - Why this matters for legal research
        
        EXTRACTED INFORMATION:
        
        FACTS:
        {facts}
        
        ISSUES:
        {issues}
        
        HOLDING:
        {holding}
        
        REASONING:
        {reasoning}
        
        RATIO DECIDENDI:
        {ratio}
        
        LEGAL MEMO:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    
    try:
        state["final_memo"] = invoke_with_retry(chain, {
            "doc_type": doc_type,
            "jurisdiction": state.get("document_metadata", {}).get("jurisdiction", "Unknown"),
            "facts": state.get("extracted_facts", ""),
            "issues": state.get("extracted_issues", ""),
            "holding": state.get("extracted_holding", ""),
            "reasoning": state.get("extracted_reasoning", ""),
            "ratio": state.get("extracted_ratio", "")
        })
        print("✅ Legal memo synthesized successfully")
    except Exception as e:
        print(f"❌ Synthesis error: {e}")
        state["final_memo"] = "Error synthesizing memo"
        state["errors"].append(f"Synthesis failed: {str(e)}")
    
    state["processing_time"]["synthesis"] = time.time() - start_time
    return state

In [9]:
class EvaluationScore(BaseModel):
    """Structured evaluation output"""
    faithfulness_score: int = Field(description="Score 0-25: No hallucinations")
    coverage_score: int = Field(description="Score 0-25: Captures key points")
    precision_score: int = Field(description="Score 0-25: Legally accurate framing")
    clarity_score: int = Field(description="Score 0-25: Professional writing")
    total_score: int = Field(description="Sum of all scores")
    strengths: List[str] = Field(description="What the brief does well")
    weaknesses: List[str] = Field(description="Areas for improvement")
    judge_feedback: str = Field(description="Overall assessment")

def evaluate_summary(state):
    """FIXED: Evaluation with safe dictionary access"""
    import time
    start_time = time.time()
    print("⚖️  [Judge] Evaluating legal memo quality...")
    
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    import json
    
    prompt = ChatPromptTemplate.from_template(
    """
    You are a senior judge evaluating a law clerk's brief.
    
    Grade strictly and independently for each document:
    
    1. Faithfulness (0-25):
       - Check EVERY claim against the original
       - Deduct 5 points per hallucination
       - Compare names, dates, holdings exactly
    
    2. Coverage (0-25):
       - Did it miss the main legal issue? -10 points
       - Missing procedural history? -5 points
       - Missing ratio decidendi? -5 points
    
    3. Precision (0-25):
       - Is "Whether..." statement clear? +5
       - Are legal terms used correctly? +5
       - Is the holding specific? +5
    
    4. Clarity (0-25):
       - Professional structure? +10
       - Readable formatting? +5
       - Proper citations? +5
    
    IMPORTANT: 
    - Scores must reflect THIS document's quality
    - Don't use default scores (18, 20, 15, 22)
    - Actually count the issues you find
    
    Return ONLY this JSON:
    {{
        "faithfulness_score": [your calculated score],
        "coverage_score": [your calculated score],
        "precision_score": [your calculated score],
        "clarity_score": [your calculated score],
        "total_score": [sum of above],
        "strengths": ["specific strength 1", "specific strength 2"],
        "weaknesses": ["specific weakness 1", "specific weakness 2"],
        "judge_feedback": "2-3 sentences about THIS specific document"
    }}
    
    ORIGINAL DOCUMENT (first 15000 chars):
    {original_text}
    
    CLERK'S MEMO:
    {generated_memo}
    """
)
    
    eval_llm = get_llm(use_fallback=True, temperature=0.1)
    chain = prompt | eval_llm | StrOutputParser()
    
    try:
        response = invoke_with_retry(chain, {
            "original_text": state["document_text"][:15000],
            "generated_memo": state["final_memo"]
        })
        
        # Clean JSON
        clean_response = response.strip()
        if clean_response.startswith("```json"):
            clean_response = clean_response[7:]
        if clean_response.startswith("```"):
            clean_response = clean_response[3:]
        if clean_response.endswith("```"):
            clean_response = clean_response[:-3]
        clean_response = clean_response.strip()
        
        try:
            result = json.loads(clean_response)
        except json.JSONDecodeError:
            print(f"⚠️  JSON parse failed, using estimated scores")
            result = {
                "faithfulness_score": 15,
                "coverage_score": 15,
                "precision_score": 15,
                "clarity_score": 15,
                "total_score": 60
            }
        
        # Safe access with defaults
        state["evaluation_score"] = {
            "faithfulness_score": int(result.get("faithfulness_score", 0)),
            "coverage_score": int(result.get("coverage_score", 0)),
            "precision_score": int(result.get("precision_score", 0)),
            "clarity_score": int(result.get("clarity_score", 0)),
            "total_score": int(result.get("total_score", 0)),
            "strengths": result.get("strengths", []),
            "weaknesses": result.get("weaknesses", []),
            "judge_feedback": result.get("judge_feedback", "No feedback provided")
        }
        
        print(f"✅ Evaluation complete - Total Score: {state['evaluation_score']['total_score']}/100")
        
    except Exception as e:
        print(f"❌ Evaluation error: {e}")
        state["evaluation_score"] = {
            "faithfulness_score": 0,
            "coverage_score": 0,
            "precision_score": 0,
            "clarity_score": 0,
            "total_score": 0,
            "strengths": [],
            "weaknesses": [],
            "judge_feedback": f"Evaluation failed: {str(e)}"
        }
        state["errors"].append(f"Evaluation failed: {str(e)}")
    
    state["processing_time"]["evaluation"] = time.time() - start_time
    return state

print("✅ Fixed functions ready to use")

✅ Fixed functions ready to use


## Build Workflow

In [10]:
# Create workflow
workflow = StateGraph(LegalDocumentState)

# Add nodes (same as before)
workflow.add_node("classify", classify_document)
workflow.add_node("extract_facts", extract_facts)
workflow.add_node("extract_issues", extract_issues)
workflow.add_node("extract_holding", extract_holding)
workflow.add_node("extract_reasoning", extract_reasoning)
workflow.add_node("extract_ratio", extract_ratio)
workflow.add_node("synthesize", synthesize_memo)
workflow.add_node("evaluate", evaluate_summary)

# Define edges
workflow.set_entry_point("classify")
workflow.add_edge("classify", "extract_facts")
workflow.add_edge("extract_facts", "extract_issues")
workflow.add_edge("extract_issues", "extract_holding")
workflow.add_edge("extract_holding", "extract_reasoning")
workflow.add_edge("extract_reasoning", "extract_ratio")
workflow.add_edge("extract_ratio", "synthesize")
workflow.add_edge("synthesize", "evaluate")
workflow.add_edge("evaluate", END)

# Compile
app = workflow.compile()
print("✅ Workflow compiled successfully!")

✅ Workflow compiled successfully!


## Enhanced Processing Function with Upload Support

In [11]:
def process_uploaded_document(file_path: str, enable_ocr: bool = OCR_ENABLED) -> Dict:
    """
    Process a legal document from uploaded file
    
    Args:
        file_path: Path to uploaded file
        enable_ocr: Whether to use OCR for scanned documents
    
    Returns:
        Dictionary with processing results
    """
    print("\n" + "="*70)
    print("🏛️  LEGAL DOCUMENT SUMMARIZER - FILE UPLOAD VERSION")
    print("="*70 + "\n")
    
    overall_start = time.time()
    
    # Step 1: Extract text from file
    try:
        extraction_result = extract_text_from_file(file_path, enable_ocr)
        document_text = extraction_result.text
        
        if not document_text or len(document_text.strip()) < 50:
            raise Exception("Insufficient text extracted from document")
        
        print(f"\n✅ Extraction complete: {len(document_text)} characters")
        print(f"   Method: {extraction_result.extraction_method}")
        if extraction_result.extraction_method == 'ocr':
            print(f"   OCR Confidence: {extraction_result.confidence:.1%}")
        
    except Exception as e:
        print(f"\n❌ Extraction failed: {e}")
        return {
            "errors": [f"Text extraction failed: {str(e)}"],
            "final_memo": "Error: Could not extract text from document"
        }
    
    # Step 2: Initialize state
    initial_state = {
        "document_text": document_text,
        "document_metadata": {
            "filename": Path(file_path).name,
            "file_type": extraction_result.file_type,
            "page_count": extraction_result.page_count
        },
        "extraction_info": {
            "method": extraction_result.extraction_method,
            "confidence": extraction_result.confidence,
            "warnings": extraction_result.warnings
        },
        "classification": "",
        "classification_confidence": 0.0,
        "extracted_facts": "",
        "extracted_issues": "",
        "extracted_holding": "",
        "extracted_reasoning": "",
        "extracted_ratio": "",
        "final_memo": "",
        "evaluation_score": {},
        "errors": [],
        "processing_time": {}
    }
    
    # Step 3: Run workflow
    try:
        print("\n🔄 Starting legal analysis workflow...\n")
        result = app.invoke(initial_state)
        
        # Calculate total time
        total_time = time.time() - overall_start
        result["processing_time"]["total"] = total_time
        
        return result
        
    except Exception as e:
        print(f"\n❌ WORKFLOW ERROR: {e}")
        initial_state["errors"].append(f"Workflow failed: {str(e)}")
        return initial_state

def display_results(result: Dict):
    """Display formatted results with extraction info"""
    print("\n" + "="*70)
    print("📊 PROCESSING COMPLETE")
    print("="*70)
    
    # Document Info
    print("\n📄 DOCUMENT INFORMATION")
    print("-" * 70)
    if result.get('document_metadata'):
        print(f"Filename: {result['document_metadata'].get('filename', 'N/A')}")
        print(f"File Type: {result['document_metadata'].get('file_type', 'N/A')}")
        print(f"Pages: {result['document_metadata'].get('page_count', 'N/A')}")
    
    if result.get('extraction_info'):
        print(f"\n📝 Extraction Method: {result['extraction_info'].get('method', 'N/A')}")
        if result['extraction_info'].get('method') == 'ocr':
            conf = result['extraction_info'].get('confidence', 0)
            print(f"   OCR Confidence: {conf:.1%}")
            if conf < 0.7:
                print("   ⚠️  Warning: Low OCR confidence may affect accuracy")
    
    # Classification
    print("\n📋 DOCUMENT CLASSIFICATION")
    print("-" * 70)
    print(f"Type: {result.get('classification', 'N/A')}")
    print(f"Confidence: {result.get('classification_confidence', 0):.1%}")
    
    # Final Memo
    print("\n📝 FINAL LEGAL MEMO")
    print("="*70)
    print(result.get('final_memo', 'No memo generated'))
    print("="*70)
    
    # Evaluation
    if result.get('evaluation_score'):
        eval_score = result['evaluation_score']
        print("\n⚖️  QUALITY EVALUATION")
        print("-" * 70)
        print(f"Faithfulness : {eval_score.get('faithfulness_score', 'N/A')}/25")
        print(f"Coverage     : {eval_score.get('coverage_score', 'N/A')}/25")
        print(f"Precision    : {eval_score.get('precision_score', 'N/A')}/25")
        print(f"Clarity      : {eval_score.get('clarity_score', 'N/A')}/25")
        print("-" * 70)
        print(f"TOTAL SCORE  : {eval_score.get('total_score', 'N/A')}/100")
        print("-" * 70)
        print(f"\n💭 FEEDBACK: {eval_score.get('judge_feedback', 'N/A')}")
    
    # Performance
    if result.get('processing_time'):
        print("\n⏱️  PERFORMANCE")
        print("-" * 70)
        total = result['processing_time'].get('total', 0)
        print(f"Total Processing Time: {total:.2f}s")
    
    # Errors
    if result.get('errors'):
        print("\n❌ ERRORS")
        print("-" * 70)
        for error in result['errors']:
            print(f"  • {error}")
    
    print("\n" + "="*70)

## Save Results Function

In [12]:
def save_results(result: Dict, original_filename: str, output_dir: str = OUTPUT_FOLDER):
    """
    Save processing results to files
    
    Creates:
    - JSON file with full results
    - TXT file with readable memo
    - Metadata file with extraction info
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = Path(original_filename).stem
    
    output_base = os.path.join(output_dir, f"{base_name}_{timestamp}")
    
    # 1. Save full JSON results
    json_path = f"{output_base}_full.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Full results saved: {json_path}")
    
    # 2. Save readable memo
    memo_path = f"{output_base}_memo.txt"
    with open(memo_path, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("LEGAL MEMORANDUM\n")
        f.write("="*70 + "\n\n")
        
        # Document info
        f.write(f"Source Document: {original_filename}\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        if result.get('extraction_info'):
            f.write(f"Extraction Method: {result['extraction_info'].get('method', 'N/A')}\n")
            if result['extraction_info'].get('method') == 'ocr':
                conf = result['extraction_info'].get('confidence', 0)
                f.write(f"OCR Confidence: {conf:.1%}\n")
        
        f.write(f"Document Type: {result.get('classification', 'N/A')}\n")
        
        if result.get('evaluation_score'):
            score = result['evaluation_score'].get('total_score', 'N/A')
            f.write(f"Quality Score: {score}/100\n")
        
        f.write("\n" + "="*70 + "\n\n")
        
        # Memo content
        f.write(result.get('final_memo', 'No memo generated'))
        
        f.write("\n\n" + "="*70 + "\n")
        f.write("END OF MEMORANDUM\n")
        f.write("="*70 + "\n")
    
    print(f"📄 Memo saved: {memo_path}")
    
    # 3. Save metadata
    meta_path = f"{output_base}_metadata.json"
    metadata = {
        "original_file": original_filename,
        "processed_at": datetime.now().isoformat(),
        "document_metadata": result.get('document_metadata', {}),
        "extraction_info": result.get('extraction_info', {}),
        "classification": result.get('classification', 'N/A'),
        "classification_confidence": result.get('classification_confidence', 0),
        "evaluation_score": result.get('evaluation_score', {}),
        "processing_time": result.get('processing_time', {}),
        "errors": result.get('errors', [])
    }
    
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"📋 Metadata saved: {meta_path}")
    print(f"\n✅ All results saved to: {output_dir}")
    
    return {
        "json": json_path,
        "memo": memo_path,
        "metadata": meta_path
    }

## Example Usage: Process Uploaded File

In [13]:
# ============================================
# OPTION 1: Process a file directly
# ============================================

# Replace with your file path
file_path = "/kaggle/input/datasets/zara26928/uploads/C.A.775_2015.pdf"

# Process the document
result = process_uploaded_document(
    file_path=file_path,
    enable_ocr=True  # Set to False to disable OCR
)

# Display results
display_results(result)

# Save results
saved_files = save_results(result, Path(file_path).name)
print(f"\n📁 Access your files at:")
for file_type, path in saved_files.items():
    print(f"   {file_type}: {path}")


🏛️  LEGAL DOCUMENT SUMMARIZER - FILE UPLOAD VERSION


📄 Processing: C.A.775_2015.pdf
   File type: .pdf
   ✅ Extracted 13800 characters from 7 pages

✅ Extraction complete: 13800 characters
   Method: text

🔄 Starting legal analysis workflow...

📋 [Classifier] Analyzing document type...
✅ Classification: court_opinion (confidence: 0.85)
📝 [Facts Agent] Extracting key facts...
✅ Facts extracted successfully
⚖️  [Issues Agent] Identifying legal issues...
✅ Issues identified successfully
🔨 [Holding Agent] Extracting court's decision...
✅ Holding extracted successfully
🧠 [Reasoning Agent] Analyzing legal reasoning...
✅ Reasoning extracted successfully
📜 [Ratio Agent] Identifying binding legal principle...
✅ Ratio decidendi identified successfully
📝 [Synthesizer] Creating final legal memo...
✅ Legal memo synthesized successfully
⚖️  [Judge] Evaluating legal memo quality...
⚠️  JSON parse failed, using estimated scores
✅ Evaluation complete - Total Score: 60/100

📊 PROCESSING COMPLETE

📄 DO

## Interactive File Upload Widget (Optional)

In [ ]:
# ============================================
# OPTION 2: Use Jupyter file upload widget
# ============================================

from IPython.display import display, HTML
from ipywidgets import FileUpload, Button, VBox, Output

# Create upload widget
upload_widget = FileUpload(
    accept='.pdf,.docx,.txt,.png,.jpg,.jpeg',
    multiple=False,
    description='Upload Document'
)

process_button = Button(description='Process Document', button_style='success')
output = Output()

def on_process_clicked(b):
    output.clear_output()
    
    with output:
        if not upload_widget.value:
            print("❌ Please upload a file first")
            return
        
        # Get uploaded file (works in Kaggle)
        uploaded_file = upload_widget.value[0]
        filename = uploaded_file["name"]
        file_content = uploaded_file["content"]
        
        # Save file
        temp_path = os.path.join(UPLOAD_FOLDER, filename)
        with open(temp_path, "wb") as f:
            f.write(file_content)
        
        print(f"📤 Processing uploaded file: {filename}\n")
        
        # Process document
        result = process_uploaded_document(temp_path, enable_ocr=True)
        
        # Display results
        display_results(result)
        
        # Save results
        saved_files = save_results(result, filename)

process_button.on_click(on_process_clicked)

display(VBox([upload_widget, process_button, output]))

print("\n📤 Upload your legal document above and click 'Process Document'")
print("   Supported formats: PDF, DOCX, TXT, PNG, JPG")
print("   OCR will be applied automatically for scanned documents")


📤 Upload your legal document above and click 'Process Document'
   Supported formats: PDF, DOCX, TXT, PNG, JPG
   OCR will be applied automatically for scanned documents


## Batch Processing Multiple Files

In [ ]:
def batch_process_directory(directory_path: str, enable_ocr: bool = True):
    """
    Process all legal documents in a directory
    """
    supported_extensions = {'.pdf', '.docx', '.txt', '.png', '.jpg', '.jpeg'}
    
    directory = Path(directory_path)
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\n📁 Found {len(files)} documents to process\n")
    
    results = []
    
    for i, file_path in enumerate(files, 1):
        print(f"\n{'='*70}")
        print(f"Processing {i}/{len(files)}: {file_path.name}")
        print(f"{'='*70}")
        
        try:
            result = process_uploaded_document(str(file_path), enable_ocr)
            saved = save_results(result, file_path.name)
            
            results.append({
                "filename": file_path.name,
                "status": "success",
                "score": result.get('evaluation_score', {}).get('total_score', 0),
                "saved_files": saved
            })
            
        except Exception as e:
            print(f"\n❌ Failed to process {file_path.name}: {e}")
            results.append({
                "filename": file_path.name,
                "status": "failed",
                "error": str(e)
            })
    
    # Summary
    print(f"\n\n{'='*70}")
    print("BATCH PROCESSING SUMMARY")
    print(f"{'='*70}")
    
    successful = [r for r in results if r['status'] == 'success']
    failed = [r for r in results if r['status'] == 'failed']
    
    print(f"✅ Successful: {len(successful)}/{len(files)}")
    print(f"❌ Failed: {len(failed)}/{len(files)}")
    
    if successful:
        avg_score = sum(r['score'] for r in successful) / len(successful)
        print(f"\n📊 Average Quality Score: {avg_score:.1f}/100")
    
    return results

# Example usage:
# results = batch_process_directory("./legal_documents", enable_ocr=True)